# ETH perpetual 15m Pine V1 audit

**TL;DR:** V9 improves the consumed final-preholdout ETH result to +30.22 bp/trade after the frozen 20 bp cost, but week-block p=0.17 and the mean turns negative after removing the largest winner. Actual 10m comparison, regime blocks, and a cost-underwater break-even warning keep V9 research-only; V10/V11 are post-selection hypotheses.

In [1]:
from pathlib import Path
import json, pandas as pd
ROOT=Path.cwd(); EXP=ROOT/'experiments/active/exp-pine-eth-15m-v1'; R=EXP/'results'
validation=json.loads((R/'validation.json').read_text())
summary=json.loads((R/'summary.json').read_text())
stats=json.loads((R/'statistical_tests.json').read_text())
intrabar=json.loads((R/'intrabar_3m_reconciliation.json').read_text())
robustness=json.loads((R/'robustness_checks.json').read_text())
docker_smoke=json.loads((R/'docker_offline_smoke.json').read_text())
v11=json.loads((R/'v11_long_only_summary.json').read_text())
control_sensitivity=json.loads((R/'control_seed_sensitivity.json').read_text())
path_risk=json.loads((R/'path_risk_bootstrap.json').read_text())
judgment=json.loads((R/'pine_judgment_development_manifest.json').read_text())
exit_anatomy=json.loads((R/'exit_anatomy.json').read_text())
actual_timeframe=json.loads((R/'actual_10m_vs_15m.json').read_text())
regime=json.loads((R/'regime_stability.json').read_text())
judgment_capacity=json.loads((R/'judgment_feasibility.json').read_text())
stateful_gate=json.loads((R/'stateful_gate_static_vs_dynamic.json').read_text())
pine_static=json.loads((R/'pine_static_contract.json').read_text())
split=pd.read_csv(R/'split_summary.csv')
risk=pd.read_csv(R/'risk_grid.csv')
assert validation['status']=='pass' and summary['holdout_consumed'] is False
print(f"{validation['counts']['checks']} checks pass; holdout consumed: 0")

52 checks pass; holdout consumed: 0


## Context & Methods

Signal close at t, entry at open(t+1); time splits only; 20 bp round-trip cost; exact ETH-month × HK 6h × ATR-quintile controls with unique starts; week-clustered inference.

In [2]:
cols=['variant','period','trades','project_net_bp_per_trade','monetary_profit_factor','return_percent','max_drawdown_15m_percent']
split.loc[split.variant.isin(['v8_eth_baseline','v9_locked','v10_volume_hypothesis']),cols].round(4)

,variant,period,trades,project_net_bp_per_trade,monetary_profit_factor,return_percent,max_drawdown_15m_percent
0,v8_eth_baseline,discovery_2023,181,-14.8593,0.9637,-4.4446,26.0100
1,v8_eth_baseline,confirmation_2024,195,23.6563,1.3534,44.9082,21.7861
2,v8_eth_baseline,final_preholdout_2025_202602,212,-28.2872,0.8257,-16.0602,28.7199
9,v9_locked,discovery_2023,83,52.0077,1.9211,70.5056,26.4646
10,v9_locked,confirmation_2024,83,141.3550,2.7478,109.2548,12.1922
11,v9_locked,final_preholdout_2025_202602,110,30.2234,1.3660,22.8183,20.0670
24,v10_volume_hypothesis,discovery_2023,62,94.7247,2.9848,100.7618,20.5321
25,v10_volume_hypothesis,confirmation_2024,64,192.4357,3.5852,117.9982,11.4325
26,v10_volume_hypothesis,final_preholdout_2025_202602,77,41.2182,1.4049,18.2403,13.5040


In [3]:
pd.DataFrame({
 'metric':['V9 candidate bp','matched control bp','excess bp','week signflip p','absolute CI low','absolute CI high'],
 'value':[stats['matched_control']['mean_candidate_net_bp'],stats['matched_control']['mean_control_net_bp'],stats['matched_control']['mean_excess_bp'],stats['week_block_signflip']['p_value'],stats['week_bootstrap_absolute']['ci95_low_bp'],stats['week_bootstrap_absolute']['ci95_high_bp']]})

,metric,value
0,V9 candidate bp,30.223421
1,matched control bp,0.795305
2,excess bp,29.428116
3,week signflip p,0.170383
4,absolute CI low,-61.869744
5,absolute CI high,214.135095


In [4]:
stats['profit_concentration']

{'trades': 110,
 'positive_trades': 11,
 'positive_trade_fraction': 0.1,
 'total_unit_net_bp': 3324.576294070858,
 'top1_share_of_net': 1.1562743080961315,
 'top3_share_of_net': 2.722526776826515,
 'top5_share_of_net': 3.4978739604287408,
 'top1_share_of_positive_sum': 0.26741914497626124,
 'top3_share_of_positive_sum': 0.6296566288259976,
 'mean_without_top1_bp': -4.766475780447011,
 'mean_without_top3_bp': -53.52029615083845,
 'reverse_trades': 11,
 'reverse_total_unit_net_bp': 14374.895090180715,
 'stop_trades': 99,
 'stop_total_unit_net_bp': -11050.318796109857,
 'positive_months': 6,
 'months_with_trades': 14,
 'median_monthly_net_bp': -38.03390955320029}

In [5]:
risk.loc[risk.period.eq('final_preholdout_2025_202602'), ['risk_percent','return_percent','max_drawdown_15m_percent','mean_leverage','max_leverage']].round(4)

,risk_percent,return_percent,max_drawdown_15m_percent,mean_leverage,max_leverage
2,0.50,12.1382,10.6305,0.2867,0.5742
5,0.75,17.6824,15.4883,0.4300,0.8613
8,1.00,22.8183,20.0670,0.5733,1.1484
11,1.25,27.5098,24.3843,0.7166,1.4356
14,1.50,31.7270,28.4565,0.8600,1.7227
17,2.00,38.6521,35.9254,1.1466,2.2969


## Decision assertions

The notebook deliberately asserts the failures as well as arithmetic validity.

In [6]:
v9=summary['v9_final_preholdout']
assert v9['project_net_bp_per_trade'] > 0
assert stats['week_block_signflip']['p_value'] >= 0.01
assert stats['week_bootstrap_absolute']['ci95_low_bp'] < 0
assert stats['profit_concentration']['mean_without_top1_bp'] < 0
assert intrabar['data_quality']['holdout_rows_read'] == 0
assert intrabar['same_15m_exit_parent_count'] == 110
assert intrabar['exact_exit_price_count'] == 110
assert robustness['final_preholdout_rows_read'] == 0
assert robustness['selection_adjusted_feature_test']['selection_adjusted_p_value'] >= 0.01
assert docker_smoke['status'] == 'pass' and docker_smoke['pinned_docker_recipe_built'] is False
assert v11['profit_concentration']['mean_without_top1_bp'] < 0
assert all(row['fraction_assignment_seeds_with_p_below_0p01'] == 0 for row in control_sensitivity['variants'])
assert path_risk['arms'][0]['drawdown_q95_percent'] < path_risk['arms'][2]['drawdown_q95_percent']
assert judgment['training_eligible'] is False and judgment['lr_fitted'] is False
assert exit_anatomy['break_even_cost_semantics']['locked_stop_project_net_bp'] == -10.0
assert actual_timeframe['variants']['V9_15m']['summary']['project_net_bp_per_trade'] < 0
assert regime['absolute_net_equal_block_test']['one_sided_p_value'] >= 0.01
assert judgment_capacity['overall_positive_events_per_feature'] < 1.0
assert stateful_gate['static_top_decile_filtering_valid_for_l2'] is False
assert pine_static['status'] == 'pass' and pine_static['official_pine_compiler_run'] is False
assert summary['tradingview_parity_passed'] is False
print('Point estimate positive; robustness and parity gates correctly remain failed.')

Point estimate positive; robustness and parity gates correctly remain failed.
